<a href="https://colab.research.google.com/github/junseok-jay/2026-1_CV/blob/main/HW%233.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 2. 실험 환경

In [3]:
# ============================================================
# 실험 환경 확인
# ============================================================
# 현재 코드가 실행되는 환경의 OS, Python 버전, 주요 라이브러리 버전,
# 그리고 GPU 사용 가능 여부를 확인한다.
# 이 출력 결과는 보고서의 "실험 환경" 항목에 사용한다.

import platform
import torch
import torchvision
import sklearn
import numpy
import pandas
import matplotlib

# 운영체제 정보 출력
print("OS:", platform.platform())

# Python 버전 출력
print("Python:", platform.python_version())

# PyTorch 및 관련 라이브러리 버전 출력
print("PyTorch:", torch.__version__)
print("Torchvision:", torchvision.__version__)

# 실험에서 사용하는 주요 라이브러리 버전 출력
print("scikit-learn:", sklearn.__version__)
print("NumPy:", numpy.__version__)
print("Pandas:", pandas.__version__)
print("Matplotlib:", matplotlib.__version__)

# 현재 사용 가능한 device 확인
# CUDA 사용 가능 시 GPU, 아니면 CPU를 사용한다.
print("Device:", torch.device("cuda" if torch.cuda.is_available() else "cpu"))

# CUDA 사용 가능 시 GPU 이름 출력
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

OS: Linux-6.6.122+-x86_64-with-glibc2.35
Python: 3.12.13
PyTorch: 2.10.0+cu128
Torchvision: 0.25.0+cu128
scikit-learn: 1.6.1
NumPy: 2.0.2
Pandas: 2.2.2
Matplotlib: 3.10.0
Device: cuda
GPU: Tesla T4


In [4]:
# ============================================================
# Random Seed 고정 및 Device 설정
# ============================================================
# 실험 결과가 실행할 때마다 달라지는 것을 줄이기 위해 random seed를 고정한다.
# 또한 GPU 사용 가능 여부를 확인하여 학습에 사용할 device를 설정한다.

import torch
import numpy as np
import random

def set_seed(seed=42):
    # Python 내장 random 모듈의 seed 고정
    random.seed(seed)

    # NumPy random seed 고정
    np.random.seed(seed)

    # PyTorch CPU 연산의 random seed 고정
    torch.manual_seed(seed)

    # PyTorch GPU 연산의 random seed 고정
    # CUDA를 사용하지 않는 환경에서는 큰 영향이 없지만,
    # GPU 사용 시 재현성을 높이기 위해 함께 설정한다.
    torch.cuda.manual_seed_all(seed)

# 전체 실험에서 사용할 seed를 42로 고정
set_seed(42)

# GPU 사용 가능 시 cuda, 아니면 cpu 사용
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 현재 학습에 사용할 device 출력
print("device:", device)

device: cuda


# dataset loader

In [5]:
# ============================================================
# Fashion-MNIST Dataset Loader
# ============================================================
# Fashion-MNIST 데이터셋을 불러와 train_loader와 valid_loader 형태로 반환한다.
# Fashion-MNIST는 28x28 grayscale image이며, 총 10개의 class를 가진다.
#
# 주의:
# 여기서 valid_dataset은 변수명은 validation이지만,
# train=False를 사용하므로 Fashion-MNIST의 공식 t10k/test split이다.
# 즉, 보고서에는 "공식 t10k split을 validation/evaluation 용도로 사용하였다"

from torchvision import datasets, transforms

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from torch.utils.data import TensorDataset, DataLoader

def load_fashion_mnist_dataset(batch_size=128):
    # 이미지에 적용할 전처리 과정을 정의한다.
    transform = transforms.Compose([
        # PIL image를 PyTorch Tensor로 변환한다.
        # pixel 값은 0~255에서 0~1 범위로 변환된다.
        transforms.ToTensor(),

        # grayscale image이므로 channel이 1개이다.
        # 평균 0.5, 표준편차 0.5로 정규화하여 값의 범위를 대략 -1~1로 만든다.
        transforms.Normalize((0.5,), (0.5,)),

        # MLP는 2차원 이미지가 아니라 1차원 벡터를 입력으로 받는다.
        # 따라서 1x28x28 이미지를 784차원 벡터로 펼친다.
        transforms.Lambda(lambda x: x.view(-1))
    ])

    # Fashion-MNIST 공식 train split을 불러온다.
    # train=True는 60,000개의 학습 데이터를 의미한다.
    train_dataset = datasets.FashionMNIST(
        root="./data",
        train=True,
        download=True,
        transform=transform
    )

    # Fashion-MNIST 공식 t10k/test split을 불러온다.
    # train=False는 10,000개의 test 데이터를 의미한다.
    # 현재 코드에서는 이를 valid_loader 이름으로 사용한다.
    valid_dataset = datasets.FashionMNIST(
        root="./data",
        train=False,
        download=True,
        transform=transform
    )

    # train_loader는 학습 데이터셋을 mini-batch 단위로 제공한다.
    # shuffle=True로 설정하여 매 epoch마다 데이터 순서를 섞는다.
    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True
    )

    # valid_loader는 평가용 데이터셋을 mini-batch 단위로 제공한다.
    # 평가에서는 데이터 순서를 섞을 필요가 없으므로 shuffle=False로 둔다.
    valid_loader = DataLoader(
        valid_dataset,
        batch_size=batch_size,
        shuffle=False
    )

    # Fashion-MNIST 이미지는 28x28이므로 flatten 후 입력 차원은 784이다.
    input_size = 784

    # Fashion-MNIST는 10개 class로 구성된다.
    num_classes = 10

    # 학습용 loader, 평가용 loader, 입력 차원, class 수를 반환한다.
    return train_loader, valid_loader, input_size, num_classes

In [6]:
# ============================================================
# Digits Dataset Loader
# ============================================================
# scikit-learn에서 제공하는 Digits 데이터셋을 불러와
# train_loader와 valid_loader 형태로 반환한다.
#
# Digits Dataset:
# - 8x8 grayscale 숫자 이미지
# - 입력 차원: 8 x 8 = 64
# - class 수: 10
# - Fashion-MNIST보다 데이터 크기가 작아 빠르게 실험 가능

from sklearn.datasets import load_digits

def load_digits_dataset(batch_size=64):
    # scikit-learn 내장 Digits 데이터셋 로드
    digits = load_digits()

    # X는 입력 이미지 데이터, y는 정답 label이다.
    # digits.data는 이미 8x8 이미지를 64차원 벡터로 펼친 형태이다.
    X = digits.data
    y = digits.target

    # 각 feature를 표준화한다.
    # 평균 0, 표준편차 1에 가깝게 변환하여 학습 안정성을 높인다.
    scaler = StandardScaler()
    X = scaler.fit_transform(X)

    # 전체 데이터를 train set과 validation set으로 분리한다.
    # test_size=0.2는 전체 데이터 중 20%를 validation으로 사용한다는 뜻이다.
    # random_state=42는 데이터 분할 결과를 재현 가능하게 만든다.
    # stratify=y는 class 비율이 train/valid에 비슷하게 유지되도록 한다.
    X_train, X_valid, y_train, y_valid = train_test_split(
        X,
        y,
        test_size=0.2,
        random_state=42,
        stratify=y
    )

    # NumPy 배열을 PyTorch Tensor로 변환한다.
    # 입력 X는 float32, label y는 class index이므로 long 타입을 사용한다.
    X_train = torch.tensor(X_train, dtype=torch.float32)
    X_valid = torch.tensor(X_valid, dtype=torch.float32)
    y_train = torch.tensor(y_train, dtype=torch.long)
    y_valid = torch.tensor(y_valid, dtype=torch.long)

    # TensorDataset은 입력 데이터와 정답 label을 하나의 dataset으로 묶는다.
    # DataLoader는 dataset을 mini-batch 단위로 모델에 공급한다.
    # train_loader는 학습용이므로 shuffle=True로 매 epoch마다 순서를 섞는다.
    train_loader = DataLoader(
        TensorDataset(X_train, y_train),
        batch_size=batch_size,
        shuffle=True
    )

    # valid_loader는 평가용이므로 shuffle=False로 둔다.
    valid_loader = DataLoader(
        TensorDataset(X_valid, y_valid),
        batch_size=batch_size,
        shuffle=False
    )

    # Digits 데이터는 8x8 이미지이므로 입력 차원은 64이다.
    input_size = 64

    # 숫자 0~9 분류 문제이므로 class 수는 10이다.
    num_classes = 10

    # 학습용 loader, 검증용 loader, 입력 차원, class 수 반환
    return train_loader, valid_loader, input_size, num_classes

In [7]:
# ============================================================
# make_moons Dataset Loader
# ============================================================
# make_moons는 실험 B에서 활성화 함수 비교에 사용하는
# 2차원 비선형 이진 분류 데이터셋이다.

from sklearn.datasets import make_moons

def load_moons_dataset(batch_size=64):
    # make_moons 데이터 생성
    # n_samples: 전체 데이터 개수
    # noise: 데이터에 추가할 잡음 정도
    # random_state: 재현성을 위한 seed
    X, y = make_moons(
        n_samples=2000,
        noise=0.25,
        random_state=42
    )

    # 입력 feature 표준화
    scaler = StandardScaler()
    X = scaler.fit_transform(X)

    # train / validation 분리
    X_train, X_valid, y_train, y_valid = train_test_split(
        X,
        y,
        test_size=0.2,
        random_state=42,
        stratify=y
    )

    # PyTorch Tensor로 변환
    X_train = torch.tensor(X_train, dtype=torch.float32)
    X_valid = torch.tensor(X_valid, dtype=torch.float32)
    y_train = torch.tensor(y_train, dtype=torch.long)
    y_valid = torch.tensor(y_valid, dtype=torch.long)

    # DataLoader 생성
    train_loader = DataLoader(
        TensorDataset(X_train, y_train),
        batch_size=batch_size,
        shuffle=True
    )

    valid_loader = DataLoader(
        TensorDataset(X_valid, y_valid),
        batch_size=batch_size,
        shuffle=False
    )

    # make_moons는 입력 차원 2, class 수 2
    return train_loader, valid_loader, 2, 2

In [8]:
# ============================================================
# MLP Model Definition
# ============================================================
# 실험 A/B/C에서 공통으로 사용할 MLP 모델을 정의한다.
# activation 인자를 통해 ReLU, LeakyReLU, Sigmoid를 선택할 수 있다.
#
# 모델 구조:
# input_size → 512 → 256 → 128 → num_classes
#
# 각 hidden layer의 activation 값(self.a1, self.a2, self.a3)을 저장해두면
# 실험 B에서 Layer별 Activation 분포와 Dead ReLU 비율을 분석할 수 있다.

import torch.nn as nn

class MLP(nn.Module):
    def __init__(self, input_size, num_classes, activation="relu"):
        super().__init__()

        # 각 hidden layer의 activation 값을 저장하기 위한 변수
        # forward가 실행되면 self.a1, self.a2, self.a3에 실제 activation 값이 저장된다.
        self.a1 = None
        self.a2 = None
        self.a3 = None

        # Fully Connected Layer 정의
        # 입력 데이터 크기에 따라 input_size가 달라진다.
        # Digits: input_size=64
        # Fashion-MNIST: input_size=784
        # make_moons: input_size=2
        self.fc1 = nn.Linear(input_size, 512)
        self.fc2 = nn.Linear(512, 256)
        self.fc3 = nn.Linear(256, 128)
        self.fc4 = nn.Linear(128, num_classes)

        # activation function 선택
        # 실험 B에서는 이 부분만 바꾸어 ReLU, LeakyReLU, Sigmoid를 비교한다.
        if activation == "relu":
            self.act1 = nn.ReLU()
            self.act2 = nn.ReLU()
            self.act3 = nn.ReLU()

        elif activation == "leaky_relu":
            # LeakyReLU는 음수 입력에 대해서도 작은 기울기를 허용한다.
            # 따라서 ReLU의 Dead ReLU 문제를 완화할 수 있다.
            self.act1 = nn.LeakyReLU(negative_slope=0.01)
            self.act2 = nn.LeakyReLU(negative_slope=0.01)
            self.act3 = nn.LeakyReLU(negative_slope=0.01)

        elif activation == "sigmoid":
            # Sigmoid는 출력을 0~1 사이로 압축한다.
            # 값이 0 또는 1 근처로 포화되면 gradient vanishing이 발생할 수 있다.
            self.act1 = nn.Sigmoid()
            self.act2 = nn.Sigmoid()
            self.act3 = nn.Sigmoid()


        # 현재 모델에서 사용하는 activation 이름 저장
        self.activation = activation

    def forward(self, x):
        # 첫 번째 hidden layer 계산
        # Linear → Activation
        self.a1 = self.act1(self.fc1(x))

        # 두 번째 hidden layer 계산
        self.a2 = self.act2(self.fc2(self.a1))

        # 세 번째 hidden layer 계산
        self.a3 = self.act3(self.fc3(self.a2))

        # 출력층 계산
        # 출력층에는 activation을 적용하지 않는다.
        # CrossEntropyLoss는 logits를 입력으로 받기 때문이다.
        out = self.fc4(self.a3)

        return out

    def get_activations(self):
        # 마지막 forward 과정에서 저장된 hidden layer activation 값을 반환한다.
        # 이 함수는 학습용이 아니라, activation 분포 시각화와 Dead ReLU 비율 계산용이다.
        return {
            "Layer1": self.a1,
            "Layer2": self.a2,
            "Layer3": self.a3
        }

In [9]:
print(MLP(input_size=64, num_classes=10, activation="relu"))

MLP(
  (fc1): Linear(in_features=64, out_features=512, bias=True)
  (fc2): Linear(in_features=512, out_features=256, bias=True)
  (fc3): Linear(in_features=256, out_features=128, bias=True)
  (fc4): Linear(in_features=128, out_features=10, bias=True)
  (act1): ReLU()
  (act2): ReLU()
  (act3): ReLU()
)


In [10]:
# ============================================================
# Evaluation Function
# ============================================================
# validation set 또는 evaluation set에서 모델의 accuracy를 계산하는 함수이다.
# 학습이 아니라 평가만 수행하므로 model.eval()과 torch.no_grad()를 사용한다.

from sklearn.metrics import accuracy_score

def evaluate(model, valid_loader):
    # 모델을 평가 모드로 전환한다.
    # Dropout, BatchNorm 같은 layer가 있을 경우 학습 모드와 평가 모드의 동작이 달라진다.
    model.eval()

    # 전체 batch의 예측값과 실제 label을 저장할 list
    all_preds = []
    all_labels = []

    # 평가 단계에서는 gradient 계산이 필요 없다.
    # torch.no_grad()를 사용하면 메모리 사용량이 줄고 평가 속도가 빨라진다.
    with torch.no_grad():
        for X, y in valid_loader:
            # 입력 데이터와 label을 현재 device로 이동한다.
            X = X.to(device)
            y = y.to(device)

            # forward propagation
            # 모델의 출력값은 class별 score, 즉 logits이다.
            outputs = model(X)

            # 가장 큰 logit 값을 가진 class를 예측 class로 선택한다.
            preds = torch.argmax(outputs, dim=1)

            # accuracy_score 계산을 위해 GPU tensor를 CPU로 옮긴 뒤 NumPy 배열로 변환한다.
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(y.cpu().numpy())

    # 전체 validation 데이터에 대한 accuracy 계산
    return accuracy_score(all_labels, all_preds)

In [11]:
# ============================================================
# Optimizer 생성 함수
# ============================================================
# optimizer_type에 따라 학습에 사용할 optimizer를 생성한다.
# 실험 C에서는 이 함수를 통해 SGD, SGD+Momentum, Adam을 바꿔가며 비교한다.

def get_optimizer(model, optimizer_type, lr):
    # SGD: 가장 기본적인 gradient descent 기반 optimizer
    # 현재 gradient 방향만 이용해 parameter를 업데이트한다.
    if optimizer_type == "sgd":
        optimizer = torch.optim.SGD(
            model.parameters(),
            lr=lr
        )

    # SGD + Momentum
    # 이전 gradient 방향을 일부 누적하여 업데이트에 반영한다.
    # 이를 통해 SGD보다 더 빠르게 수렴하거나 진동을 줄일 수 있다.
    elif optimizer_type == "momentum":
        optimizer = torch.optim.SGD(
            model.parameters(),
            lr=lr,
            momentum=0.01
        )

    # Adam
    # parameter별로 adaptive learning rate를 적용하는 optimizer이다.
    # 일반적으로 초기 수렴 속도가 빠르고 안정적인 편이다.
    elif optimizer_type == "adam":
        optimizer = torch.optim.Adam(
            model.parameters(),
            lr=lr
        )

    # 잘못된 optimizer_type이 입력되었을 때 명확한 에러를 출력한다.
    else:
        raise ValueError("optimizer_type은 'sgd', 'momentum', 'adam' 중 하나여야 합니다.")

    return optimizer

In [12]:
import torch.optim as optim
import torch.nn.functional as F

def train_model(
    model,
    train_loader,
    valid_loader,
    loss_type="cross_entropy",   # 사용할 손실 함수: "cross_entropy" 또는 "mse"
    optimizer_type="adam",       # 사용할 optimizer: "sgd", "momentum", "adam"
    lr=0.001,                    # 초기 learning rate
    epochs=30,                   # 전체 학습 epoch 수
    use_scheduler=False          # learning rate scheduler 사용 여부
):
    # 모델을 GPU로 이동
    model = model.to(device)

    ## 손실 함수 설정
    # CrossEntropyLoss는 내부적으로 softmax를 포함하므로 별도 softmax를 적용하지 않음
    if loss_type == "cross_entropy":
        loss_fn = nn.CrossEntropyLoss()

    # MSELoss는 확률값과 one-hot label을 비교하기 위해 softmax를 따로 적용해야 함
    elif loss_type == "mse":
        loss_fn = nn.MSELoss()
    #
    else:
        raise ValueError(f"Invalid loss type: {loss_type}")

    ## optimizer 생성
    # 실제 learning rate는 optimizer.param_groups[0]["lr"]에 저장됨
    optimizer = get_optimizer(model, optimizer_type=optimizer_type, lr=lr)

    ## scheduler 설정
    # ExponentialLR은 epoch마다 optimizer 내부 lr을 gamma비율 만큼 감소시킴
    scheduler = None
    if use_scheduler:
        scheduler = optim.lr_scheduler.ExponentialLR(optimizer, gamma=0.9)

    # 학습 과정 기록 수치들
    history = {
        "loss": [],
        "train_acc": [],
        "valid_acc": [],
        "grad_norm": [],
        "lr": []
    }

    # epoch 단위 학습 반복
    for epoch in range(epochs):
        # 모델을 학습 모드로 전환
        # ?
        model.train()

        # 해당 epoch동안 누적할 값 초기화
        total_loss = 0
        correct = 0
        total = 0
        total_grad_norm = 0

        # mini-batch 단위 학습
        for X, y in train_loader:
            # batch 데이터를 device로 이동
            X = X.to(device)
            y = y.to(device)

            # 이전 batch에서 계산된 gradient 초기화
            optimizer.zero_grad()

            # foward propagation
            outputs = model(X)

            # CrossEntropyLoss 사용 시 logits와 class index를 그대로 사용
            if loss_type == "cross_entropy":
                loss = loss_fn(outputs, y)

            # MSELoss 사용 시 softmax 확률값과 one-hot label을 비교
            elif loss_type == "mse":
                probs = torch.softmax(outputs, dim=1)
                y_onehot = F.one_hot(y, num_classes=outputs.shape[1]).float()
                loss = loss_fn(probs, y_onehot)

            ## backward propagation
            # 각 parameter에 대한 gradient 계산
            loss.backward()

            # gradient norm 계산
            # layer output이 아니라 parameter gradient의 크기를 측정함
            grad_norm = 0
            for param in model.parameters():
                if param.grad is not None:
                    grad_norm += param.grad.norm(2).item()

            total_grad_norm += grad_norm

            # optimizer가 현재 learning rate를 사용하여 parameter 업데이트
            optimizer.step()

            # loss 누적
            total_loss += loss.item()

            # train accuracy 계산용 예측값
            preds = torch.argmax(outputs, dim=1)
            correct += (preds == y).sum().item()
            total += y.size(0)

        # epoch별 평균 loss
        avg_loss = total_loss / len(train_loader)
        # epoch별 train accuracy
        train_acc = correct / total
        # validation accuracy 계산
        # evaluate() 내부에서 model.eval(), torch.no_grad() 사용
        valid_acc = evaluate(model, valid_loader)

        # epoch별 평균 gradient norm
        avg_grad_norm = total_grad_norm / len(train_loader)

        # 현재 optimizer가 실제로 사용하는 learning rate
        # 주의: 함수 인자인 lr은 고정값이고, scheduler는 이 값을 직접 바꾸지 않음
        current_lr = optimizer.param_groups[0]["lr"]

        # 기록 저장
        history["loss"].append(avg_loss)
        history["train_acc"].append(train_acc)
        history["valid_acc"].append(valid_acc)
        history["grad_norm"].append(avg_grad_norm)
        history["lr"].append(optimizer.param_groups[0]["lr"])

        # scheduler가 있으면 epoch 종료 후 learning rate 감소
        # 다음 epoch부터 감소된 lr이 optimizer.step()에 사용됨
        if scheduler is not None:
          scheduler.step()

        # 현재 epoch 결과 출력
        print(
            f"[loss={loss_type}, opt={optimizer_type}, lr={current_lr}] "
            f"Epoch {epoch+1}/{epochs} | "
            f"Loss: {avg_loss:.4f} | "
            f"Train Acc: {train_acc:.4f} | "
            f"Valid Acc: {valid_acc:.4f} | "
            f"Grad Norm: {avg_grad_norm:.4f}"
        )
    # 학습된 모델과 기록 반환
    return model, history

# A

In [13]:
# ============================================================
# Experiment A: Loss Function 비교
# ============================================================
# 실험 A는 손실 함수가 학습 결과에 미치는 영향을 비교하는 실험이다.
#
# 비교 대상:
# 1. CrossEntropy Loss
# 2. MSE Loss with Softmax
#
# 고정 조건:
# - Activation Function: ReLU
# - Optimizer: Adam
# - Learning Rate: 0.001
#
# 변경 조건:
# - Loss Function만 변경한다.

def run_experiment_A(
    dataset_name,
    train_loader,
    valid_loader,
    input_size,
    num_classes,
    epochs
):
    # 각 loss function별 실험 결과를 저장할 dictionary
    results = {}

    # 실험 A에서는 activation, optimizer, learning rate를 고정한다.
    fixed_activation = "relu"
    fixed_optimizer = "adam"
    fixed_lr = 0.001

    # CrossEntropy Loss와 MSE Loss를 순서대로 실험한다.
    for loss_type in ["cross_entropy", "mse"]:
        print("=" * 70)
        print(f"Experiment A | Dataset: {dataset_name} | Loss: {loss_type}")
        print("=" * 70)

        # 같은 조건에서 비교하기 위해 매 실험마다 seed를 동일하게 고정한다.
        # 이를 통해 초기 가중치와 데이터 shuffle 영향을 최대한 동일하게 맞춘다.
        set_seed(42)

        # 동일한 MLP 구조를 생성한다.
        # 실험 A에서는 activation을 ReLU로 고정한다.
        model = MLP(
            input_size=input_size,
            num_classes=num_classes,
            activation=fixed_activation
        )

        # 모델 학습 수행
        # loss_type만 cross_entropy 또는 mse로 바뀌고,
        # optimizer, learning rate, activation은 동일하게 유지된다.
        model, history = train_model(
            model=model,
            train_loader=train_loader,
            valid_loader=valid_loader,
            loss_type=loss_type,
            optimizer_type=fixed_optimizer,
            lr=fixed_lr,
            epochs=epochs,
            use_scheduler=False
        )

        # 해당 loss function의 학습된 모델과 학습 기록을 저장한다.
        results[loss_type] = {
            "model": model,
            "history": history
        }

    # 실험 조건과 결과를 하나의 dictionary로 반환한다.
    # 이후 그래프 시각화와 정량 비교표 생성에 사용된다.
    return {
        "experiment": "A",
        "dataset": dataset_name,
        "fixed_activation": fixed_activation,
        "fixed_optimizer": fixed_optimizer,
        "fixed_lr": fixed_lr,
        "results": results
    }

In [14]:
# ============================================================
# Experiment A Visualization
# ============================================================
# 실험 A의 결과를 그래프로 시각화하는 함수이다.
#
# 비교 대상:
# - CrossEntropy Loss
# - MSE Loss with Softmax
#
# 출력 그래프:
# 1. Loss vs Epoch
# 2. Validation Accuracy vs Epoch
# 3. Gradient Norm vs Epoch

import matplotlib.pyplot as plt

def plot_experiment_A(result):
    # 현재 실험에 사용된 데이터셋 이름을 가져온다.
    dataset = result["dataset"]

    # CrossEntropy 실험의 history
    ce = result["results"]["cross_entropy"]["history"]

    # MSE with Softmax 실험의 history
    mse = result["results"]["mse"]["history"]

    # ------------------------------------------------------------
    # 1. Loss vs Epoch 그래프
    # ------------------------------------------------------------
    # 두 손실 함수의 train loss 감소 양상을 비교한다.
    # 단, CrossEntropy와 MSE는 loss scale이 다르므로
    # loss 값의 크기 자체를 직접 비교하기보다는 감소 추세를 중심으로 해석한다.
    plt.figure(figsize=(8, 5))
    plt.plot(ce["loss"], label="CrossEntropy")
    plt.plot(mse["loss"], label="MSE with Softmax")
    plt.title(f"{dataset} - Loss / Epoch")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.legend()
    plt.grid(True)
    plt.show()

    # ------------------------------------------------------------
    # 2. Validation Accuracy vs Epoch 그래프
    # ------------------------------------------------------------
    # 각 loss function이 validation accuracy를 얼마나 빠르게 높이는지 확인한다.
    # 수렴 속도와 최종 성능 비교에 사용된다.
    plt.figure(figsize=(8, 5))
    plt.plot(ce["valid_acc"], label="CrossEntropy")
    plt.plot(mse["valid_acc"], label="MSE with Softmax")
    plt.title(f"{dataset} - Accuracy / Epoch")
    plt.xlabel("Epoch")
    plt.ylabel("Valid Accuracy")
    plt.legend()
    plt.grid(True)
    plt.show()

    # ------------------------------------------------------------
    # 3. Gradient Norm vs Epoch 그래프
    # ------------------------------------------------------------
    # gradient norm은 parameter gradient의 전체 크기를 의미한다.
    # 값이 너무 작으면 vanishing gradient 가능성이 있고,
    # 값이 너무 크면 학습이 불안정하거나 exploding gradient 가능성이 있다.
    plt.figure(figsize=(8, 5))
    plt.plot(ce["grad_norm"], label="CrossEntropy")
    plt.plot(mse["grad_norm"], label="MSE with Softmax")
    plt.title(f"{dataset} - Gradient Norm / Epoch")
    plt.xlabel("Epoch")
    plt.ylabel("Gradient Norm")
    plt.legend()
    plt.grid(True)
    plt.show()

In [15]:
# ============================================================
# Convergence Epoch 계산 함수
# ============================================================
# validation accuracy가 특정 기준값(threshold)에 처음 도달한 epoch를 찾는다.
#
# 예:
# threshold = 0.95
# valid_acc = [0.80, 0.91, 0.94, 0.96, 0.97]
# → 4번째 epoch에서 처음 0.95 이상이 되므로 return 4
#
# 주의:
# Python list index는 0부터 시작하지만,
# epoch 번호는 보통 1부터 세기 때문에 i + 1을 반환한다.

def find_convergence_epoch(acc_list, threshold):
    # epoch별 accuracy를 순서대로 확인한다.
    for i, acc in enumerate(acc_list):
        # accuracy가 기준값 이상이 되는 첫 번째 지점을 찾는다.
        if acc >= threshold:
            # index는 0부터 시작하므로 실제 epoch 번호는 i + 1이다.
            return i + 1

    # 끝까지 기준값에 도달하지 못한 경우 None 반환
    return None

In [16]:
# ============================================================
# Experiment A 정량 비교표 생성 함수
# ============================================================
# 실험 A의 결과를 표 형태로 정리한다.
#
# 비교 대상:
# - CrossEntropy
# - MSE with Softmax
#
# 표에 포함되는 항목:
# - Final Valid Accuracy
# - Best Valid Accuracy
# - Best Epoch
# - Minimum Loss
# - Final Gradient Norm
# - Convergence Epoch

import pandas as pd

def make_experiment_A_table(result, threshold):
    # 표의 각 행을 저장할 list
    rows = []

    # 실험 A에서 비교한 두 손실 함수에 대해 반복
    for loss_type, label in [
        ("cross_entropy", "CrossEntropy"),
        ("mse", "MSE with Softmax")
    ]:
        # 해당 loss function의 학습 기록 가져오기
        hist = result["results"][loss_type]["history"]

        # validation accuracy 중 가장 높은 값
        best_acc = max(hist["valid_acc"])

        # 가장 높은 validation accuracy가 나온 epoch
        # list index는 0부터 시작하므로 +1을 해준다.
        best_epoch = hist["valid_acc"].index(best_acc) + 1

        # 하나의 실험 결과를 dictionary 형태로 저장
        rows.append({
            # 사용한 데이터셋 이름
            "Dataset": result["dataset"],

            # 사용한 손실 함수 이름
            "Loss Function": label,

            # 실험 A에서 고정한 activation function
            "Fixed Activation": result["fixed_activation"],

            # 실험 A에서 고정한 optimizer
            "Fixed Optimizer": result["fixed_optimizer"],

            # 실험 A에서 고정한 learning rate
            "Learning Rate": result["fixed_lr"],

            # 마지막 epoch에서의 validation accuracy
            "Final Valid Accuracy": hist["valid_acc"][-1],

            # 전체 epoch 중 가장 높았던 validation accuracy
            "Best Valid Accuracy": best_acc,

            # best validation accuracy가 나온 epoch
            "Best Epoch": best_epoch,

            # 학습 중 가장 낮았던 train loss
            # 단, CrossEntropy와 MSE는 loss scale이 다르므로
            # loss 값 자체의 크기를 직접 비교하기보다는 감소 경향을 중심으로 해석한다.
            "Minimum Loss": min(hist["loss"]),

            # 마지막 epoch에서의 gradient norm
            # 학습 후반부의 gradient 크기를 확인하는 지표이다.
            "Final Gradient Norm": hist["grad_norm"][-1],

            # validation accuracy가 threshold에 처음 도달한 epoch
            # threshold에 도달하지 못하면 None이 들어간다.
            "Convergence Epoch": find_convergence_epoch(hist["valid_acc"], threshold)
        })

    # rows list를 pandas DataFrame으로 변환하여 반환
    return pd.DataFrame(rows)

# B

In [17]:
# ============================================================
# Experiment B: Activation Function 비교
# ============================================================
# 실험 B는 활성화 함수가 학습 결과에 미치는 영향을 비교하는 실험이다.
#
# 비교 대상:
# 1. ReLU
# 2. LeakyReLU
# 3. Sigmoid
#
# 고정 조건:
# - Loss Function: CrossEntropy
# - Optimizer: Adam
# - Learning Rate: 0.01
#
# 변경 조건:
# - Activation Function만 변경한다.
#
# 분석 목적:
# - ReLU에서 Dead ReLU가 발생하는지 확인
# - LeakyReLU가 Dead ReLU 문제를 완화하는지 확인
# - Sigmoid에서 vanishing gradient 문제가 나타나는지 확인

def run_experiment_B(
    train_loader,
    valid_loader,
    input_size,
    num_classes,
    epochs=300
):
    # activation function별 실험 결과를 저장할 dictionary
    results = {}

    # 실험 B에서는 loss, optimizer, learning rate를 고정한다.
    fixed_loss = "cross_entropy"
    fixed_optimizer = "adam"
    fixed_lr = 0.01

    # ReLU, LeakyReLU, Sigmoid를 순서대로 실험한다.
    for activation in ["relu", "leaky_relu", "sigmoid"]:
        print("=" * 70)
        print(f"Experiment B | Activation: {activation}")
        print("=" * 70)

        # 같은 조건에서 activation function만 비교하기 위해 seed를 고정한다.
        set_seed(42)

        # activation 인자를 통해 MLP 내부 활성화 함수를 변경한다.
        # 나머지 layer 구조는 동일하다.
        model = MLP(
            input_size=input_size,
            num_classes=num_classes,
            activation=activation
        )

        # 모델 학습 수행
        # activation만 다르고 loss, optimizer, learning rate는 동일하다.
        model, history = train_model(
            model=model,
            train_loader=train_loader,
            valid_loader=valid_loader,
            loss_type=fixed_loss,
            optimizer_type=fixed_optimizer,
            lr=fixed_lr,
            epochs=epochs,
            use_scheduler=False
        )

        # 해당 activation function의 학습된 모델과 학습 기록 저장
        results[activation] = {
            "model": model,
            "history": history
        }

    # 실험 조건과 결과를 하나의 dictionary로 반환한다.
    # 이후 그래프 시각화, Dead ReLU 비율 계산, 정량 비교표 생성에 사용된다.
    return {
        "experiment": "B",
        "dataset": "make_moons",
        "fixed_loss": fixed_loss,
        "fixed_optimizer": fixed_optimizer,
        "fixed_lr": fixed_lr,
        "results": results
    }

In [18]:
# ============================================================
# Experiment B Visualization
# ============================================================
# 실험 B의 결과를 그래프로 시각화하는 함수이다.
#
# 비교 대상:
# - ReLU
# - LeakyReLU
# - Sigmoid
#
# 출력 그래프:
# 1. Loss vs Epoch
# 2. Validation Accuracy vs Epoch
# 3. Gradient Norm vs Epoch
#
# 이 그래프를 통해 활성화 함수에 따라
# 학습 속도, 정확도 변화, gradient 흐름이 어떻게 달라지는지 확인한다.

# import matplotlib.pyplot as plt

def plot_experiment_B(result):
    # 그래프 legend에 표시할 activation function 이름
    labels = {
        "relu": "ReLU",
        "leaky_relu": "LeakyReLU",
        "sigmoid": "Sigmoid"
    }

    # 시각화할 metric 목록
    # loss       : 학습 오차 변화
    # valid_acc  : validation accuracy 변화
    # grad_norm  : gradient 크기 변화
    for metric, ylabel in [
        ("loss", "Loss"),
        ("valid_acc", "Valid Accuracy"),
        ("grad_norm", "Gradient Norm")
    ]:
        # 그래프 크기 설정
        plt.figure(figsize=(8, 5))

        # activation function별 학습 기록을 하나의 그래프에 표시
        for activation, value in result["results"].items():
            plt.plot(
                value["history"][metric],
                label=labels[activation]
            )

        # 그래프 제목 및 축 이름 설정
        plt.title(f"Experiment B - {ylabel} / Epoch")
        plt.xlabel("Epoch")
        plt.ylabel(ylabel)

        # 범례, grid 표시
        plt.legend()
        plt.grid(True)

        # 그래프 출력
        plt.show()

In [19]:
# ============================================================
# Dead ReLU Ratio 계산 함수
# ============================================================
# 각 layer의 activation 값 중 0에 가까운 값의 비율을 계산한다.
#
# Dead ReLU란:
# - ReLU에서 입력이 음수이면 출력이 0이 된다.
# - 특정 neuron의 출력이 계속 0에 가까우면 gradient가 흐르지 않아
#   학습에 거의 기여하지 못할 수 있다.
#
# 이 함수는 실험 B에서 activation function별 Dead ReLU 정도를 비교하는 데 사용한다.

def calculate_dead_ratio(model, data_loader):
    # 모델을 평가 모드로 전환한다.
    # activation 값을 확인하는 용도이므로 학습 모드가 아니라 평가 모드로 둔다.
    model.eval()

    # data_loader에서 첫 번째 batch만 가져온다.
    # 전체 데이터가 아니라 일부 batch를 사용해 layer별 activation 상태를 확인한다.
    X, y = next(iter(data_loader))

    # 입력 데이터를 현재 device로 이동한다.
    X = X.to(device)

    # activation 분석은 학습이 아니므로 gradient 계산을 하지 않는다.
    with torch.no_grad():
        # forward를 한 번 실행해야 model 내부의 self.a1, self.a2, self.a3에
        # activation 값이 저장된다.
        _ = model(X)

        # 저장된 layer별 activation 값을 가져온다.
        activations = model.get_activations()

    # layer별 dead ratio를 저장할 dictionary
    ratios = {}

    # 각 layer의 activation 값에 대해 dead ratio 계산
    for layer_name, values in activations.items():
        # Tensor를 NumPy 배열로 변환한다.
        # detach(): gradient 계산 그래프에서 분리
        # cpu(): GPU tensor를 CPU로 이동
        # numpy(): NumPy 배열로 변환
        values = values.detach().cpu().numpy()

        # activation 값이 0에 매우 가까운 비율을 계산한다.
        # 1e-6보다 작은 값을 dead activation으로 간주한다.
        ratios[layer_name] = np.mean(np.abs(values) < 1e-6)

    # 예: {"Layer1": 0.42, "Layer2": 0.58, "Layer3": 0.61}
    return ratios

In [20]:
# ============================================================
# Experiment B 정량 비교표 생성 함수
# ============================================================
# 실험 B의 결과를 표 형태로 정리한다.
#
# 비교 대상:
# - ReLU
# - LeakyReLU
# - Sigmoid
#
# 표에 포함되는 항목:
# - Activation Function
# - Dead Ratio
# - Final Valid Accuracy
# - Best Valid Accuracy
# - Best Epoch
# - Final Gradient Norm
# - Convergence Epoch
#
# 주의:
# 현재 MLP 모델은 Layer1, Layer2, Layer3의 activation을 저장한다.
# 따라서 Layer3 Dead Ratio도 함께 표에 넣는 것이 좋다.

# import pandas as pd

def make_experiment_B_table(result, data_loader, threshold=0.90):
    # 표의 각 행을 저장할 list
    rows = []

    # activation function별 결과를 하나씩 꺼내서 처리한다.
    for activation, value in result["results"].items():
        # 해당 activation function의 학습 기록
        hist = value["history"]

        # 해당 activation function으로 학습된 모델
        model = value["model"]

        # layer별 Dead ReLU 비율 계산
        # 예: {"Layer1": 0.42, "Layer2": 0.55, "Layer3": 0.60}
        dead = calculate_dead_ratio(model, data_loader)

        # validation accuracy 중 가장 높은 값
        best_acc = max(hist["valid_acc"])

        # best validation accuracy가 나온 epoch
        # list index는 0부터 시작하므로 +1을 해준다.
        best_epoch = hist["valid_acc"].index(best_acc) + 1

        # 하나의 activation function에 대한 결과를 dictionary로 저장
        rows.append({
            # 사용한 activation function
            "Activation": activation,

            # 실험 B에서 고정한 loss function
            "Fixed Loss": result["fixed_loss"],

            # 실험 B에서 고정한 optimizer
            "Fixed Optimizer": result["fixed_optimizer"],

            # 실험 B에서 고정한 learning rate
            "Learning Rate": result["fixed_lr"],

            # 각 hidden layer에서 activation 값이 0에 가까운 비율
            # ReLU의 경우 Dead ReLU 정도를 확인하는 지표로 사용된다.
            "Layer1 Dead Ratio": dead["Layer1"],
            "Layer2 Dead Ratio": dead["Layer2"],
            "Layer3 Dead Ratio": dead["Layer3"],

            # 마지막 epoch에서의 validation accuracy
            "Final Valid Accuracy": hist["valid_acc"][-1],

            # 전체 epoch 중 가장 높았던 validation accuracy
            "Best Valid Accuracy": best_acc,

            # best validation accuracy가 나온 epoch
            "Best Epoch": best_epoch,

            # 마지막 epoch에서의 gradient norm
            # 학습 후반부 gradient 흐름을 확인하는 지표이다.
            "Final Gradient Norm": hist["grad_norm"][-1],

            # validation accuracy가 threshold에 처음 도달한 epoch
            # 도달하지 못하면 None이 들어간다.
            "Convergence Epoch": find_convergence_epoch(hist["valid_acc"], threshold)
        })

    # rows list를 pandas DataFrame으로 변환하여 반환
    return pd.DataFrame(rows)

# C

In [21]:
# ============================================================
# Experiment C: Optimizer 및 Learning Rate 비교
# ============================================================
# 실험 C는 optimizer와 learning rate가 학습 결과에 미치는 영향을 비교하는 실험이다.
#
# 비교 대상 optimizer:
# 1. SGD
# 2. SGD + Momentum
# 3. Adam
#
# 비교 대상 learning rate:
# - 0.1
# - 0.01
# - 0.001
#
# 고정 조건:
# - Activation Function: ReLU
# - Loss Function: CrossEntropy
#
# 변경 조건:
# - Optimizer
# - Learning Rate
#
# 추가 조건:
# - ExponentialLR scheduler를 사용하여 epoch마다 learning rate를 감소시킨다.

def run_experiment_C(
    dataset_name,
    train_loader,
    valid_loader,
    input_size,
    num_classes,
    epochs=50
):
    # optimizer와 learning rate 조합별 실험 결과를 저장할 dictionary
    results = {}

    # 실험 C에서는 activation과 loss를 고정한다.
    fixed_activation = "relu"
    fixed_loss = "cross_entropy"

    # 비교할 optimizer 목록
    optimizer_list = ["sgd", "momentum", "adam"]

    # 비교할 learning rate 목록
    lr_list = [0.1, 0.01, 0.001]

    # optimizer와 learning rate의 모든 조합을 실험한다.
    for optimizer_type in optimizer_list:
        for lr in lr_list:
            # 결과 저장용 key
            # 예: "sgd_lr0.1", "adam_lr0.001"
            key = f"{optimizer_type}_lr{lr}"

            print("=" * 70)
            print(f"Experiment C | Optimizer: {optimizer_type} | LR: {lr}")
            print("=" * 70)

            # 같은 조건에서 optimizer와 lr만 비교하기 위해 seed를 고정한다.
            set_seed(42)

            # 동일한 MLP 구조 생성
            # 실험 C에서는 activation을 ReLU로 고정한다.
            model = MLP(
                input_size=input_size,
                num_classes=num_classes,
                activation=fixed_activation
            )

            # 모델 학습 수행
            # optimizer_type과 lr만 바뀌고, loss와 activation은 고정된다.
            # use_scheduler=True이므로 ExponentialLR이 적용된다.
            model, history = train_model(
                model=model,
                train_loader=train_loader,
                valid_loader=valid_loader,
                loss_type=fixed_loss,
                optimizer_type=optimizer_type,
                lr=lr,
                epochs=epochs,
                use_scheduler=True
            )

            # 해당 optimizer/lr 조합의 결과 저장
            results[key] = {
                "optimizer": optimizer_type,
                "lr": lr,
                "model": model,
                "history": history
            }

    # 실험 조건과 전체 결과 반환
    # 이후 그래프 시각화와 정량 비교표 생성에 사용된다.
    return {
        "experiment": "C",
        "dataset": dataset_name,
        "fixed_activation": fixed_activation,
        "fixed_loss": fixed_loss,
        "scheduler": "ExponentialLR(gamma=0.9)",
        "results": results
    }

In [22]:
# ============================================================
# Experiment C Visualization
# ============================================================
# 실험 C의 결과를 그래프로 시각화하는 함수이다.
#
# 비교 대상:
# - SGD
# - SGD + Momentum
# - Adam
#
# 각 optimizer에 대해 learning rate 0.1, 0.01, 0.001을 비교한다.
#
# 출력 그래프:
# 1. Loss vs Epoch
# 2. Validation Accuracy vs Epoch
# 3. Gradient Norm vs Epoch
#
# 이 그래프를 통해 optimizer와 learning rate에 따라
# 수렴 속도, 진동 여부, 학습 안정성이 어떻게 달라지는지 확인한다.

# import matplotlib.pyplot as plt

def plot_experiment_C(result):
    # 시각화할 metric 목록
    # loss       : train loss 변화
    # valid_acc  : validation accuracy 변화
    # grad_norm  : gradient 크기 변화
    for metric, ylabel in [
        ("loss", "Loss"),
        ("valid_acc", "Valid Accuracy"),
        ("grad_norm", "Gradient Norm")
    ]:
        # 그래프 크기 설정
        plt.figure(figsize=(10, 6))

        # optimizer + learning rate 조합별 결과를 하나의 그래프에 표시
        # 예: sgd_lr0.1, momentum_lr0.01, adam_lr0.001
        for key, value in result["results"].items():
            plt.plot(
                value["history"][metric],
                label=key
            )

        # 그래프 제목 및 축 이름 설정
        plt.title(f"Experiment C - {ylabel} / Epoch")
        plt.xlabel("Epoch")
        plt.ylabel(ylabel)

        # 범례와 grid 표시
        plt.legend()
        plt.grid(True)

        # 그래프 출력
        plt.show()

In [23]:
# ============================================================
# Experiment C 정량 비교표 생성 함수
# ============================================================
# 실험 C의 결과를 표 형태로 정리한다.
#
# 비교 대상:
# - SGD
# - SGD + Momentum
# - Adam
#
# 각 optimizer에 대해 learning rate 0.1, 0.01, 0.001을 비교한다.
#
# 표에 포함되는 항목:
# - Optimizer
# - Learning Rate
# - Final Valid Accuracy
# - Best Valid Accuracy
# - Best Epoch
# - Minimum Loss
# - Final Gradient Norm
# - Loss Stability
# - Convergence Epoch

# import pandas as pd

def make_experiment_C_table(result, threshold=0.95):
    # 표의 각 행을 저장할 list
    rows = []

    # optimizer + learning rate 조합별 결과를 하나씩 처리한다.
    for key, value in result["results"].items():
        # 해당 조합의 학습 기록
        hist = value["history"]

        # validation accuracy 중 가장 높은 값
        best_acc = max(hist["valid_acc"])

        # best validation accuracy가 나온 epoch
        # list index는 0부터 시작하므로 +1을 해준다.
        best_epoch = hist["valid_acc"].index(best_acc) + 1

        # 학습 후반부 loss 안정성 계산
        # 마지막 10 epoch loss의 표준편차를 사용한다.
        # 값이 작을수록 후반부 loss 변동이 작고 안정적이라고 해석할 수 있다.
        loss_stability = np.std(hist["loss"][-10:])

        # 하나의 optimizer/lr 조합에 대한 결과를 dictionary로 저장
        rows.append({
            # 사용한 optimizer
            "Optimizer": value["optimizer"],

            # 사용한 learning rate
            "Learning Rate": value["lr"],

            # 실험 C에서 고정한 activation function
            "Fixed Activation": result["fixed_activation"],

            # 실험 C에서 고정한 loss function
            "Fixed Loss": result["fixed_loss"],

            # 실험 C에서 적용한 learning rate scheduler
            "Scheduler": result["scheduler"],

            # 마지막 epoch에서의 validation accuracy
            "Final Valid Accuracy": hist["valid_acc"][-1],

            # 전체 epoch 중 가장 높았던 validation accuracy
            "Best Valid Accuracy": best_acc,

            # best validation accuracy가 나온 epoch
            "Best Epoch": best_epoch,

            # 학습 중 가장 낮았던 train loss
            "Minimum Loss": min(hist["loss"]),

            # 마지막 epoch에서의 gradient norm
            # optimizer별 학습 후반부 gradient 흐름을 비교하는 데 사용한다.
            "Final Gradient Norm": hist["grad_norm"][-1],

            # 마지막 10 epoch의 loss 표준편차
            # optimizer별 학습 안정성을 비교하는 지표로 사용한다.
            "Loss Stability(std last 10)": loss_stability,

            # validation accuracy가 threshold에 처음 도달한 epoch
            # 도달하지 못하면 None이 들어간다.
            "Convergence Epoch": find_convergence_epoch(hist["valid_acc"], threshold)
        })

    # rows list를 pandas DataFrame으로 변환하여 반환
    return pd.DataFrame(rows)

# GO

In [ ]:
# ============================================================
# Experiment A 실행부
# ============================================================
# 실험 A는 손실 함수 비교 실험이다.
#
# 실행 대상 데이터셋:
# 1. Digits Dataset
# 2. Fashion-MNIST
#
# 비교 대상 손실 함수:
# - CrossEntropy
# - MSE with Softmax
#
# 각 데이터셋에 대해 실험 A를 수행한 뒤,
# Loss / Accuracy / Gradient Norm 그래프를 출력하고,
# 정량 비교표를 생성한다.

# import pandas as pd

# Digits 데이터셋 로드
# 반환값:
# - digits_train_loader: Digits 학습 데이터 loader
# - digits_valid_loader: Digits 검증 데이터 loader
# - digits_input_size: Digits 입력 차원, 64
# - digits_num_classes: class 수, 10
digits_train_loader, digits_valid_loader, digits_input_size, digits_num_classes = load_digits_dataset()

# Fashion-MNIST 데이터셋 로드
# 반환값:
# - fashion_train_loader: Fashion-MNIST 학습 데이터 loader
# - fashion_valid_loader: Fashion-MNIST 평가 데이터 loader
# - fashion_input_size: Fashion-MNIST 입력 차원, 784
# - fashion_num_classes: class 수, 10
fashion_train_loader, fashion_valid_loader, fashion_input_size, fashion_num_classes = load_fashion_mnist_dataset()

# ------------------------------------------------------------
# Digits Dataset에 대해 실험 A 수행
# ------------------------------------------------------------
# 고정 조건:
# - activation = ReLU
# - optimizer = Adam
# - learning rate = 0.001
#
# 변경 조건:
# - loss function: CrossEntropy, MSE
#
# Digits는 작은 데이터셋이므로 epochs=50으로 설정한다.
digits_result_A = run_experiment_A(
    dataset_name="Digits",
    train_loader=digits_train_loader,
    valid_loader=digits_valid_loader,
    input_size=digits_input_size,
    num_classes=digits_num_classes,
    epochs=40
)

# ------------------------------------------------------------
# Fashion-MNIST에 대해 실험 A 수행
# ------------------------------------------------------------
# Fashion-MNIST는 Digits보다 데이터가 크므로 epochs=30으로 설정한다.
fashion_result_A = run_experiment_A(
    dataset_name="Fashion-MNIST",
    train_loader=fashion_train_loader,
    valid_loader=fashion_valid_loader,
    input_size=fashion_input_size,
    num_classes=fashion_num_classes,
    epochs=50
)

# Digits 실험 A 결과 시각화
# 출력:
# - Loss vs Epoch
# - Valid Accuracy vs Epoch
# - Gradient Norm vs Epoch
plot_experiment_A(digits_result_A)

# Fashion-MNIST 실험 A 결과 시각화
plot_experiment_A(fashion_result_A)

# ------------------------------------------------------------
# 실험 A 정량 비교표 생성
# ------------------------------------------------------------
# Digits와 Fashion-MNIST의 결과표를 하나로 합친다.
#
# threshold:
# - Digits는 비교적 쉬운 데이터셋이므로 0.95 기준 사용
# - Fashion-MNIST는 더 어려운 데이터셋이므로 0.85 기준 사용
#
# Convergence Epoch는 valid accuracy가 threshold에 처음 도달한 epoch이다.
experiment_A_table = pd.concat([
    make_experiment_A_table(digits_result_A, threshold=0.95),
    make_experiment_A_table(fashion_result_A, threshold=0.85)
], ignore_index=True)

# 실험 A 최종 정량 비교표 출력
experiment_A_table

100%|██████████| 26.4M/26.4M [00:02<00:00, 9.85MB/s]
100%|██████████| 29.5k/29.5k [00:00<00:00, 172kB/s]
100%|██████████| 4.42M/4.42M [00:01<00:00, 3.19MB/s]
100%|██████████| 5.15k/5.15k [00:00<00:00, 29.6MB/s]


Experiment A | Dataset: Digits | Loss: cross_entropy
[loss=cross_entropy, opt=adam, lr=0.001] Epoch 1/40 | Loss: 1.4452 | Train Acc: 0.6625 | Valid Acc: 0.8667 | Grad Norm: 2.1641
[loss=cross_entropy, opt=adam, lr=0.001] Epoch 2/40 | Loss: 0.2613 | Train Acc: 0.9214 | Valid Acc: 0.9472 | Grad Norm: 2.2161
[loss=cross_entropy, opt=adam, lr=0.001] Epoch 3/40 | Loss: 0.1040 | Train Acc: 0.9708 | Valid Acc: 0.9611 | Grad Norm: 2.0370
[loss=cross_entropy, opt=adam, lr=0.001] Epoch 4/40 | Loss: 0.0529 | Train Acc: 0.9882 | Valid Acc: 0.9556 | Grad Norm: 1.4195
[loss=cross_entropy, opt=adam, lr=0.001] Epoch 5/40 | Loss: 0.0353 | Train Acc: 0.9910 | Valid Acc: 0.9750 | Grad Norm: 1.0948
[loss=cross_entropy, opt=adam, lr=0.001] Epoch 6/40 | Loss: 0.0156 | Train Acc: 0.9972 | Valid Acc: 0.9778 | Grad Norm: 0.5479
[loss=cross_entropy, opt=adam, lr=0.001] Epoch 7/40 | Loss: 0.0092 | Train Acc: 0.9979 | Valid Acc: 0.9750 | Grad Norm: 0.3599
[loss=cross_entropy, opt=adam, lr=0.001] Epoch 8/40 | Loss

In [ ]:
# ============================================================
# Experiment B 실행부
# ============================================================
# 실험 B는 활성화 함수 비교 실험이다.
#
# 사용 데이터셋:
# - make_moons
#
# 비교 대상 활성화 함수:
# - ReLU
# - LeakyReLU
# - Sigmoid
#
# 고정 조건:
# - Loss Function: CrossEntropy
# - Optimizer: Adam
# - Learning Rate: 0.01
#
# 분석 항목:
# - Loss 변화
# - Valid Accuracy 변화
# - Gradient Norm 변화
# - Dead ReLU Ratio

# make_moons 데이터셋 로드
# 반환값:
# - moon_train_loader: 학습 데이터 loader
# - moon_valid_loader: 검증 데이터 loader
# - moon_input_size: 입력 차원, 2
# - moon_num_classes: class 수, 2
moon_train_loader, moon_valid_loader, moon_input_size, moon_num_classes = load_moons_dataset()

# ------------------------------------------------------------
# make_moons 데이터셋에 대해 실험 B 수행
# ------------------------------------------------------------
# activation function만 ReLU, LeakyReLU, Sigmoid로 변경하고,
# 나머지 조건은 동일하게 유지한다.
#
# make_moons는 비선형 2D 분류 문제이므로,
# activation function 차이를 분석하기에 적합하다.
#
# 과제 가이드에 따라 make_moons는 200~500 epoch 정도를 사용할 수 있으므로
# 여기서는 epochs=300으로 설정한다.
result_B = run_experiment_B(
    train_loader=moon_train_loader,
    valid_loader=moon_valid_loader,
    input_size=moon_input_size,
    num_classes=moon_num_classes,
    epochs=100
)

# ------------------------------------------------------------
# 실험 B 그래프 출력
# ------------------------------------------------------------
# 출력 그래프:
# - Loss vs Epoch
# - Valid Accuracy vs Epoch
# - Gradient Norm vs Epoch
plot_experiment_B(result_B)

# ------------------------------------------------------------
# 실험 B 정량 비교표 생성
# ------------------------------------------------------------
# threshold=0.90:
# validation accuracy가 0.90 이상에 처음 도달한 epoch를
# convergence epoch로 계산한다.
#
# make_experiment_B_table 내부에서 calculate_dead_ratio()를 호출하여
# 각 activation function별 Dead Ratio도 함께 계산한다.
experiment_B_table = make_experiment_B_table(
    result_B,
    data_loader=moon_valid_loader,
    threshold=0.90
)

# 실험 B 최종 정량 비교표 출력
experiment_B_table

In [ ]:
# ============================================================
# Experiment C 실행부
# ============================================================
# 실험 C는 optimizer와 learning rate 비교 실험이다.
#
# 사용 데이터셋:
# - Digits Dataset
#
# 비교 대상 optimizer:
# - SGD
# - SGD + Momentum
# - Adam
#
# 비교 대상 learning rate:
# - 0.1
# - 0.01
# - 0.001
#
# 고정 조건:
# - Activation Function: ReLU
# - Loss Function: CrossEntropy
# - Scheduler: ExponentialLR(gamma=0.9)
#
# 분석 항목:
# - Loss 변화
# - Valid Accuracy 변화
# - Gradient Norm 변화
# - 수렴 속도
# - 학습 안정성

# ------------------------------------------------------------
# Digits Dataset에 대해 실험 C 수행
# ------------------------------------------------------------
# run_experiment_C 내부에서 optimizer와 learning rate의 모든 조합을 실험한다.
#
# 조합 예:
# - sgd_lr0.1
# - sgd_lr0.01
# - sgd_lr0.001
# - momentum_lr0.1
# - momentum_lr0.01
# - momentum_lr0.001
# - adam_lr0.1
# - adam_lr0.01
# - adam_lr0.001
#
# use_scheduler=True로 설정되어 있으므로,
# 각 epoch가 끝날 때마다 learning rate가 0.9배로 감소한다.
result_C = run_experiment_C(
    dataset_name="Digits",
    train_loader=digits_train_loader,
    valid_loader=digits_valid_loader,
    input_size=digits_input_size,
    num_classes=digits_num_classes,
    epochs=30
)

# ------------------------------------------------------------
# 실험 C 그래프 출력
# ------------------------------------------------------------
# 출력 그래프:
# - Loss vs Epoch
# - Valid Accuracy vs Epoch
# - Gradient Norm vs Epoch
#
# 여러 optimizer/lr 조합이 한 그래프에 표시되므로,
# 수렴 속도와 진동 여부를 비교할 수 있다.
plot_experiment_C(result_C)

# ------------------------------------------------------------
# 실험 C 정량 비교표 생성
# ------------------------------------------------------------
# threshold=0.95:
# validation accuracy가 0.95 이상에 처음 도달한 epoch를
# convergence epoch로 계산한다.
#
# make_experiment_C_table 내부에서 다음 지표를 정리한다.
# - Final Valid Accuracy
# - Best Valid Accuracy
# - Best Epoch
# - Minimum Loss
# - Final Gradient Norm
# - Loss Stability
# - Convergence Epoch
experiment_C_table = make_experiment_C_table(
    result_C,
    threshold=0.95
)

# 실험 C 최종 정량 비교표 출력
experiment_C_table